# Exploring the Standard Model

Written by:

* Philip Ilten (University of Cincinnati)

This notebook provides a short introduction to some of the underlying physics of the Standard Model (SM) of particle physics: conservation of momemtum, Lorentz boosts, electromagnetic coupling, and quantum chromodynamics radiation.

## Requirements

Before running this notebook, we need to set up our environment. First, we install and import the `wurlitzer` module. This allows programs that have C-like backends to write their output to the Python console. In short, this allows the output of Pythia to be displayed in this notebook.

In [ ]:
# Redirect the C output of Pythia to the notebook.
!pip install wurlitzer
from wurlitzer import sys_pipes_forever

sys_pipes_forever()

Next, we need to install the Pythia module.

In [ ]:
# Install and import the Pythia module.
!pip install pythia8mc
import pythia8mc as pythia8

Finally, we need to download and import the visualization module `vistas`.

In [ ]:
# Download the visualization module.
!wget -q -N https://gitlab.com/mcgen-ct/tutorials/-/raw/main/.full/pythia/vistas.py

# Import `vistas` module.
import vistas

We also need to set up the visualization.

In [ ]:
# If a web browser window is not opening, you can set this to a higher
# value (in seconds) and then click the printed link.
sleep = 5


# The following creates a VISTAS object and configures it.
def Viewer(pythia, show=["hard process"]):
    viewer = vistas.Vistas(pythia)
    viewer.opts = viewer.options()
    viewer.opts["length"]["scale"] = "log"
    viewer.opts["length"]["observable"] = "p.pAbs()"
    if len(show) > 0:
        viewer.opts["show"] = show
    viewer.opts["frame"] = "lab"
    viewer.opts["length"]["skew"] = 1
    viewer.opts["length"]["offset"] = 30
    return viewer

## Introduction

The Standard Model (SM) of particle physics describes what we believe to be the interactions between fundamental particles, objects that have no known substructure. The Pythia event generator implements the math behind the SM and simulates the interactions of these particles. Conceptually, using Pythia is straight forward; we tell Pythia what our initial particles and it then generates the resulting particles from these collding particles. These final particles are what would be observed by, for example, the large detectors at the European Organization for Nuclear Research (CERN) such as ATLAS, CMS, or LHCb on the Large Hadron Collider (LHC).

Let's try simulating an electron and positron colliding at a relatively low energy with Pythia. First, we need to tell Pythia what we want to simulate.

In [ ]:
# Create the Pythia simulation object.
pythia = pythia8.Pythia("", False)
pythia.readString("Print:quiet = on")
pythia.readString("Random:setSeed = on")
pythia.readString("Random:seed = 1")

# Define the initial particles.
# Beam A is an electron.
pythia.readString("Beams:idA = 11")
# Beam B is a positron.
pythia.readString("Beams:idB = -11")

# Allow Pythia to simulate low-energy collisions.
pythia.readString("PhaseSpace:mHatMin = 0.0")
pythia.readString("23:mMin = 0.0")

# Set the momentum of the initial particles.
pythia.readString("Beams:frameType = 3")
# Give the electron a momentum of 0.4 GeV along the positive z-direction.
pythia.readString("Beams:pxA = 0")
pythia.readString("Beams:pyA = 0")
pythia.readString("Beams:pzA = 0.4")
# Give the positron a momentum of 0.4 GeV along the negative z-direction.
pythia.readString("Beams:pxB = 0")
pythia.readString("Beams:pyB = 0")
pythia.readString("Beams:pzB = -0.4")

# Tell Pythia what process we want to simulate.
pythia.readString("WeakSingleBoson:ffbar2gmZ = on")

# Turn off some parts of the simulation that aren't needed yet.
pythia.readString("PDF:lepton = off")
pythia.readString("PartonLevel:ISR = off")
pythia.readString("PartonLevel:FSR = off")
pythia.readString("HadronLevel:Hadronize = off")
pythia.readString("HadronLevel:Decay = off")

# Initialize Pythia.
pythia.init()

Next we can visualize the simulation produced by Pythia. The following cell should open up a new browser window. If it does not, click the printed link.

In [ ]:
# Simulate the collision.
pythia.next()

# View the result.
viewer = Viewer(pythia)
viewer.display(sleep=sleep)

Each line represents a particle, where the direction of the line is the direction of the particle's momentum, and the length of the line is the magnitude of the particle's momentum. Use the object selection tool to inspect each particle (looks like a mouse cursor hovering over a circle). Identify the initial electron and positron in the simulation. What are the final particles that are produced?

What's going on with the particle line that connects the initial and final particles? This is the intermediate particle that Pythia simulates which describes which force this interaction is occuring. Use the selection tool to determine what type of particle this intermediate particle is. Is this particle the carrier for one of the four forces of the SM? If so, which one?

If we were to run the simulation cell above again, would we get the same result? Try this. Compare to how this simulation might be similar or different to colliding two pool balls together.

# Kinematics

In physics we use the word kinematics to describe the movement of objects. In particle physics, kinematics specifically means the momentum-related observables for a particle. Run the simulation cell a few more times. Can you observe any connection between the kinematics of the two outgoing particles for each simulation? Use the selection tool to find the numerical value for each component of the final particles' momentum ($p_x$, $p_y$, and $p_z$). Can you show what the connection is with these numbers?

We have set up our initial electron and positron to have momentum in equal but opposite directions along the $z$-axis. What happens if we change the momentum of the electron to be larger?

In [ ]:
# Change the electron a momentum along the positive z-direction.
pythia.readString("Beams:pzA = 1")
pythia.init()
viewer = Viewer(pythia)

In [ ]:
# Simulate and display.
pythia.next()
viewer.display(sleep=sleep)

The intermediate particle now covers up the initial positron so we can no longer see it. Does the connection between the momentum components of the final particles from before still hold? What about a connection between the summed momenta of the initial particles and the final particles? What about a connection between the summed energy of the initial particles and the final particles.

In special relativity there is a relation between energy and momentum, the energy-momentum relation.
$$
E^2 = (pc)^2 + (mc^2)^2
$$
This is the more complete version of the famous $E = mc^2$ equation. Practically, this means that if a particle has a given mass we can trade momentum for energy, or vice versa. We can already see this in the simulations we performed. If we have an electron and positron as final particles, the final particle momenta are the same length as the initial particle momenta. Use the selection tool and the intermediate particle to confirm that Pythia is respecting the energy-momentum relation.

Increase the momentum of the electron in the simulation cell above even more. Run a few simulations. On average does the angle between the final particles change from the lower-momentum simulation? How might this be related to the energy-momentum relation?

## Couplings

In the previous section we saw that not every simuation we ran had the same final particles. Let's see if we can find a pattern in the types of these final particles that are produced.

In [ ]:
# We need to go back to our initial set up.
pythia.readString("Beams:pzA = 0.4")
# Change this seed to a unique number if running with other groups.
pythia.readString("Random:seed = 1")
pythia.init()
viewer = Viewer(pythia)

Now, run the simulation below at a minimum twenty times (ideally more like one hundred times) and track the number of different final particle states that are produced. If working with other groups, make sure to set `Random:seed` to a unique number for each group.

In [ ]:
# Simulate and display.
pythia.next()
viewer.display(sleep=sleep)

What are the counts of the different final particle states? Which final particle state is the most common? Which is the least common? Are there any final particles states which have similar counts?

The process we are simulating occurs through an excited photon. The photon couples to a specific final particle state with a strength given by the the product of the absolute value of the electromagnetic charges of the particles.

| fermion | charge |
| --- | --- |
| $e^-$ | $-1$ |
| $\mu^-$ | $-1$ |
| $u$ | $+2/3$ |
| $d$ | $-1/3$ |

Using the table above, do the simulated number of particle counts make sense? If not, what might be missing?

## Thresholds

So far we have only seen four final particle states: $e^+e^-$, $\mu^+\mu^-$, $u\bar{u}$, and $d\bar{d}$; why aren't the other quarks and leptons that we know exist being produced? Increase the momentum of both the initial electron and positron to 1.7 GeV and run the simulation a few times. Are there any new final particle states? What happens if the electron and positron momentum is increased to 2 GeV and then 5 GeV?

In [ ]:
# We need to go back to our initial set up.
pythia.readString("Beams:pzA = 1.7")
pythia.readString("Beams:pzB = -1.7")
pythia.init()
viewer = Viewer(pythia)

In [ ]:
# Simulate and display.
pythia.next()
viewer.display(sleep=sleep)

Look up the masses (in GeV) of the $\tau$ lepton, the charm quark ($c)$, and the beauty (or bottom) quark ($b$). Is there a connection between these masses, the final particle states we observe, and the momentum of the initial electron and positron?

## Quantum Chromodynamics

In our simulation above, we have final particle states that have quarks in them, like $u \bar{u}$. But, we have never observed a quark by itself in nature; they are always part of a bound state like a proton. So what is happening in our simulation? We have turned off two important parts of the simulation. Let's turn them back on, one-by-one.

The first missing part of the simulation is particle radiation. Just like charged particles like an electron can radiate photons, quarks can radiate gluons. We can turn this back on with the following cell.

In [ ]:
# Increase the collision energy.
pythia.readString("Beams:pzA = 20")
pythia.readString("Beams:pzB = -20")
# Turn on final state radiation.
pythia.readString("PartonLevel:FSR = on")
pythia.init()
viewer = Viewer(pythia, ["hard process", "FSR"])

Run a few simulations. What has changed in the simulation? Are there any general patterns in the radiation off the quarks that is now included?

In [ ]:
# Simulate and display.
pythia.next()
viewer.display(sleep=sleep)

The second part of the simulation that is missing is the combination of quarks and gluons into bound states into hadrons. This process is called "hadronization". We can turn this part of the simulation on with the cell below.

In [ ]:
# Turn on final state radiation.
pythia.readString("HadronLevel:Hadronize = on")
pythia.init()
viewer = Viewer(pythia, [])

Run a few simulations. What are some of the hadrons that are being produced? Are there any patterns in the production of these hadrons?

In [ ]:
# Simulate and display.
pythia.next()
viewer.display(sleep=sleep)